# 🎯 RadarMap - Tactical Squad Player Simulator

This Jupyter Notebook simulates squad members in the **RadarMap** tactical Apple Watch & iOS application.
It connects to the **Firebase Realtime Database** via the `firebase-admin` Python SDK, matching the watchOS/iOS client schema:

### 📡 Protocol & Schema Features
- **Ultra-Lean 4-Element Telemetry**: Streams `[latitude, longitude, heartRate, timestamp]` (or extended 6/7-element formats).
- **Geodesic Bearing / Course Over Ground (COG)**: Computes forward geodesic heading from instantaneous velocity vector.
- **Tactical Indicators**: Places and manages Squad Orders (`watchHere`, `goHere`, `attackHere`) under `/t/{roomId}/o` (self-pruning, no shared cap) and Enemy/Environment Indicators (`infantry`, `vehicle`, `armor`, `drone`) under `/t/{roomId}/i` (shared cap, server-enforced by a Cloud Function).
- **Constant Bandwidth Adaptation**: Implements $R_{max}(P) = R_{base} \times \min(1.0, N_{threshold} / P)$ to cap aggregate server bandwidth.
- **Dead Reckoning & Delta Gating**: Optional suppression of updates when displacement $< 3.5\text{m}$ and $\Delta\text{HR} < 12\text{ BPM}$ with 7.5s fallback heartbeat.
- **Biometric Heart Rate & Downed State**: Real-time heart rate stress zones and flatline ($0.0\text{ BPM}$) KIA / downed state trigger.

### 🔑 Before you start: Firebase credentials
This notebook authenticates as a privileged server client via a **service-account JSON key** (not the iOS app's `GoogleService-Info.plist`). Install `firebase-admin` (`pip install -r requirements.txt`), download a key from the Firebase Console for the `radarmap-8adf0` project (Project Settings → Service Accounts → Generate new private key), save it locally under `credentials/` or `notebooks/credentials/` (both `.gitignore`d — never commit it), and set `CREDENTIALS_PATH` in the configuration cell below (or export `FIREBASE_CREDENTIALS` in your shell before launching Jupyter).

## ⚙️ 1. Configuration & Parameters

Configure your player callsign, target squad room, mandatory PIN, center GPS coordinates, orbit radius, and speed below.

In [ ]:
# =============================================================================
# 🎮 PLAYER & ROOM CONFIGURATION
# =============================================================================
CALLSIGN = "VIPER-1"            # Player callsign displayed on tactical radar
ROOM_NAME = "TEST"             # Room name / Squad ID (case-insensitive e.g. "ALPHA")
PIN = "1234"                        # Mandatory PIN, 4-16 digits (see ROOM_ID_HARDENING.md §2)

# GPS Center Location (Default: San Francisco, CA)
LATITUDE = 37.33233141            # Center latitude
LONGITUDE = -122.0312186         # Center longitude
ALTITUDE = None                 # Altitude in meters (optional)

# Biometrics & Movement Parameters
HEART_RATE = 115.0              # Heart rate in BPM (0.0 = Downed / KIA)
CIRCLE_RADIUS_METERS = 50.0     # Radius of circular path in meters
SPEED_MPS = 4.5                 # Movement speed in m/s (~16 km/h jog)

# Telemetry Rates & Network Settings
UPDATE_INTERVAL_SEC = 1.0       # Telemetry interval (1.0s = 1.0 Hz baseline)
DATABASE_URL = "https://radarmap-8adf0-default-rtdb.firebaseio.com"
TELEMETRY_FORMAT = "compact4"   # "compact4" ([lat,lng,hr,ts]), "compact6", "compact7", or "dict"
ENABLE_DELTA_GATING = False     # Set True to enable dead reckoning delta gating

# Firebase Credentials — service-account JSON key (firebase-admin SDK, privileged server access).
# Leave as None to fall back to the FIREBASE_CREDENTIALS environment variable; if neither is set,
# initialization fails with a clear error rather than silently using some other auth method.
CREDENTIALS_PATH = None         # e.g. "../credentials/your-service-account-key.json"

print("✅ Configuration loaded:")
print(f"   • Callsign:    {CALLSIGN}")
print(f"   • Room:        {ROOM_NAME}")
print(f"   • Center:      ({LATITUDE:.6f}, {LONGITUDE:.6f})")
print(f"   • Orbit:       Radius = {CIRCLE_RADIUS_METERS}m | Speed = {SPEED_MPS} m/s")
print(f"   • Biometrics:  Heart Rate = {HEART_RATE} BPM")
print(f"   • Schema:      Format = {TELEMETRY_FORMAT} | Interval = {UPDATE_INTERVAL_SEC}s")
print(f"   • Credentials: {CREDENTIALS_PATH or '(using FIREBASE_CREDENTIALS env var)'}")

## 🛰️ 2. Initialize Simulation Engine

Instantiates `RadarPlayerSimulator` using parameters configured above.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("."))
from player_simulator import RadarPlayerSimulator

sim = RadarPlayerSimulator(
    callsign=CALLSIGN,
    room_name=ROOM_NAME,
    pin=PIN,
    latitude=LATITUDE,
    longitude=LONGITUDE,
    altitude=ALTITUDE,
    heart_rate=HEART_RATE,
    circle_radius_meters=CIRCLE_RADIUS_METERS,
    speed_mps=SPEED_MPS,
    update_interval_sec=UPDATE_INTERVAL_SEC,
    database_url=DATABASE_URL,
    credentials_path=CREDENTIALS_PATH,
    telemetry_format=TELEMETRY_FORMAT,
    enable_delta_gating=ENABLE_DELTA_GATING
)

print(f"✅ Simulator initialized for '{sim.callsign}' (Member ID: {sim.member_id})")

## 👑 3A. Option A: Host a New Squad Room

Run this cell to create a new squad room on Firebase as **Host**.

In [3]:
sim.host_room()

[SUCCESS] Hosted room 'TEST' as 'VIPER-1' (Host: Yes, PIN: Enabled, TTL: 7 Days).


True

## 🤝 3B. Option B: Join an Existing Squad Room

Run this cell to join a squad room already created on Apple Watch or another client.

In [ ]:
sim.join_room()

## 🏃 4. Live Circular Movement & Telemetry Stream

Run this cell to stream live positional updates to the Firebase Realtime Database.
- Open the Apple Watch Simulator / app to see this player moving in real time on the tactical radar map!
- Interrupt the kernel (Stop ⏹️ button) or pass `duration_sec=30` to run for a fixed period.

In [5]:
# Run simulation (pass e.g. duration_sec=60 for a 1-minute run, or None for continuous)
sim.run_simulation(duration_sec=None)


🚀 Simulation started for player 'VIPER-1' in room 'TEST'!
📍 Center: (37.332331, -122.031219)
🔄 Radius: 50.0 m | Speed: 4.5 m/s | Interval: 1.0s
❤️ Heart Rate: 115 BPM | Format: compact4
Press Ctrl+C or Interrupt Kernel to stop...

✅ [T+6575.5s] Lat:  37.332748 | Lon: -122.031433 | Hdg: 247.7° | HR: 115 BPM | Seq: 17107

⏹️ Simulation interrupted by user.

Completed 17107 telemetry updates.


## 🚩 5. Tactical Indicators Management

RadarMap supports placing tactical markers under `/t/{roomId}/o` (squad orders) and `/t/{roomId}/i` (enemy + environment, shared cap):
- **Squad Orders**: `watchHere` (Eye), `goHere` (Arrow Down), `attackHere` (Crossed Swords)
- **Enemy Indicators**: `infantry` (Personnel), `vehicle` (Light Vehicle), `armor` (Heavy/Armored), `drone` (Air)

In [7]:
# Place a Squad Order (e.g. "Go Here" 40 meters North of center)
target_lat = LATITUDE + (40.0 / sim.METERS_PER_DEG_LAT)
target_lon = LONGITUDE
order_id = sim.place_tactical_indicator("goHere", target_lat, target_lon)

# Place an Enemy Indicator (e.g. "Armor" 60 meters East of center)
enemy_lat = LATITUDE
enemy_lon = LONGITUDE + (60.0 / (sim.METERS_PER_DEG_LAT * 0.79))
enemy_id = sim.place_tactical_indicator("armor", enemy_lat, enemy_lon)

# Fetch active indicators from server
indicators = sim.get_tactical_indicators()
print(f"\nActive indicators in room '{sim.room_name}' ({len(indicators)} total):")
for k, v in indicators.items():
    print(f"  • [{v.get('type')}] ID: {k} at ({v.get('latitude'):.6f}, {v.get('longitude'):.6f})")

[TACTICAL] Enqueued compact indicator 'goh' (ind_9fccf85e) placed by 'sim_32686168' at (37.332691, -122.031219).
[TACTICAL] Enqueued compact indicator 'arm' (ind_b474c100) placed by 'sim_32686168' at (37.332331, -122.030535).

Active indicators in room 'TEST' (0 total):


## ❤️ 6. Biometrics & Downed / KIA Simulation

Simulate physiological changes. Setting `heart_rate = 0.0` triggers the **KIA / Downed** state on connected squad watches.

In [7]:
# Simulate sprinting (Elevated HR)
sim.set_heart_rate(165.0)

# Or simulate Downed / KIA (0.0 BPM)
# sim.set_downed(True)

[BIOMETRICS] Heart rate updated to 165 BPM.


## 📈 7. Constant Bandwidth Rate Adaptation Equation

Calculates theoretical maximum update frequency $R_{max}(P)$ and update interval as squad size scales beyond the threshold ($N=12$ players).

In [ ]:
print("Player Count | Max Update Rate (Hz) | Update Interval (s) | Agg Bandwidth Multiplier")
print("-" * 78)
for p in [1, 2, 4, 8, 10, 12, 16, 20, 50, 100]:
    rate = RadarPlayerSimulator.solve_max_update_rate_hz(p)
    interval = RadarPlayerSimulator.solve_update_interval(p)
    agg_bw = p * rate
    print(f"{p:12d} | {rate:20.3f} | {interval:19.3f} | {agg_bw:24.1f} pkt/s")

## 🚪 8. Leave / Disband Room & Cleanup

Cleanly removes the member from the room or disbands the room nodes if host.

In [11]:
sim.leave_room()


[INFO] Leaving room 'TEST'...
[SUCCESS] Disbanded room 'TEST' and purged all nodes.


KeyboardInterrupt: 